# Scaling Tools: `LLMToolSelectorMiddleware`

When an agent has **many** tools, sending all of them to the model every turn is slow, costly, and error-prone (the model may pick the wrong one). `LLMToolSelectorMiddleware` uses a small LLM step to pick only the **most relevant tools** for the current question before the main model runs.

```
user query --> tool selector (LLM) --> shortlist of tools --> main model
```

In [1]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


## An agent with six tools

We register six unrelated tools but cap the agent to the **2 most relevant** per request with `max_tools=2`.

In [3]:
from langchain.agents.middleware import LLMToolSelectorMiddleware


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: sunny, 24C."


@tool
def get_stock_price(ticker: str) -> str:
    """Get the latest stock price for a ticker symbol."""
    return f"{ticker}: $123.45"


@tool
def translate_text(text: str, target_language: str) -> str:
    """Translate text into the target language."""
    return f"[{target_language}] {text}"


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another."""
    return f"{amount} {from_currency} = {amount * 1.1:.2f} {to_currency}"


@tool
def get_news(topic: str) -> str:
    """Get the latest headline for a topic."""
    return f"Headline about {topic}."


@tool
def book_flight(origin: str, destination: str) -> str:
    """Book a flight between two cities."""
    return f"Flight booked {origin} -> {destination}."


all_tools = [get_weather, get_stock_price, translate_text, convert_currency, get_news, book_flight]

agent = create_agent(
    model=llm_noreason,
    tools=all_tools,
    middleware=[
        LLMToolSelectorMiddleware(model=llm_noreason, max_tools=2),
    ],
)

### Weather question
The selector should narrow six tools down to the weather-related one, and the model calls `get_weather`.

In [4]:
resp = agent.invoke(
    {"messages": [HumanMessage(content="What's the weather in Paris right now?")]}
)
for m in resp["messages"]:
    m.pretty_print()

================================ Human Message =================================

What's the weather in Paris right now?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_382cfdee-d8a9-4621-b313-34adebadbbf0)
 Call ID: call_382cfdee-d8a9-4621-b313-34adebadbbf0
  Args:
    city: Paris
================================= Tool Message =================================
Name: get_weather

Paris: sunny, 24C.
================================== Ai Message ==================================

The weather in Paris is currently sunny with a temperature of 24°C.


### Currency question
A different question selects a different tool (`convert_currency`) - same agent, no code change.

In [6]:
resp = agent.invoke(
    {"messages": [HumanMessage(content="How much is 50 USD in EUR?")]}
)
for m in resp["messages"]:
    m.pretty_print()

================================ Human Message =================================

How much is 50 USD in EUR?
================================== Ai Message ==================================
Tool Calls:
  convert_currency (call_ce89ed75-a3cf-4854-b0a3-6b104be5052c)
 Call ID: call_ce89ed75-a3cf-4854-b0a3-6b104be5052c
  Args:
    amount: 50
    from_currency: USD
    to_currency: EUR
================================= Tool Message =================================
Name: convert_currency

50.0 USD = 55.00 EUR
================================== Ai Message ==================================

50 USD is equal to 55.00 EUR.


## Summary

- `LLMToolSelectorMiddleware` shortlists tools before the main model call.
- `max_tools` caps how many are offered; `always_include` forces certain tools to always be available.
- Keeps prompts small and improves tool-choice accuracy as your toolset grows.